In [0]:
# spark.conf.set('spark.databricks.io.cache.enabled', False)  # Not available on Serverless compute

In [0]:
from pyspark.sql.functions import *

transactions_df =(spark
                  .range(0,150000000,1,2)
                  .select('id',
                        round(rand() * 10000,2).alias('amount'),
                        (col('id') % 10).alias('country_id'),
                        (col('id') % 100).alias('store_id'),
                  )
)

transactions_df.display()

Now we'll write the data to a table

In [0]:
transactions_df.write.mode('overwrite').saveAsTable('transactions')

In [0]:
stores_df = (spark
             .range(0, 99)
             .select(
                 'id',
                 round(rand() * 100).alias('employees'),
                 (col('id') % 10).alias('country_id'),
                 expr('uuid()').alias('name')
                 
             )
)

stores_df.display()


In [0]:
stores_df.write.saveAsTable('stores')

Now let's create a lookup table that maps country_id from the data tables to an actual country name

In [0]:
countries=  [
    (0, "Italy"),
    (1, "France"),
    (2, "Spain"),
    (3, "Germany"),
    (4, "Netherlands"),
    (5, "Belgium"),
    (6, "Greece"),
    (7, "Portugal"),
    (8, "Austria"),
    (9, "Switzerland"),
    (11, "USA"),
    (12, "Canada")
]

columns =["id", "name"]
countries_df=spark.createDataFrame(data=countries, schema=columns)
countries_df.display()
countries_df.write.saveAsTable('countries')


In [0]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.databricks.adaptive.autoBroadcastJoinThreshold", -1)

In [0]:
joined_df = spark.sql(
    f"""
    SELECT
        transactions.id,
        amount,
        countries.name as country_name,
        employees,
        stores.name as store_name
    FROM
        transactions
    left join
        stores
    on
        transactions.store_id = stores.id
    left join
        countries
    on
        transactions.country_id = countries.id
    """
)

## joined_df.display()
joined_df.write.mode('overwrite').saveAsTable('transact_countries')

Broadcast **hint**... 2 segundos menos jajaja

In [0]:
joined_dfb = spark.sql(
    f"""
    SELECT
        /*+ broadcast(country_name) */
        transactions.id,
        amount,
        countries.name as country_name,
        employees,
        stores.name as store_name
    FROM
        transactions
    left join
        stores
    on
        transactions.store_id = stores.id
    left join
        countries
    on
        transactions.country_id = countries.id
    """
)

## joined_df.display()
joined_dfb.write.mode('overwrite').saveAsTable('transact_countries_t')

Por defecto , al ser spark igualmente , divide los datos por default en 200 particiones spark.sql.shuffle.partitions

In [0]:
%sql
describe detail transactions

In [0]:
%sql
OPTIMIZE transactions ZORDER by(store_id)

In [0]:
%sql
select 
  country_id,
  count(*) as total,
  AVG(amount) as avg_amount
from 
  transactions
  group by 
    country_id
    order by 
      total desc